### Reading data which is in json format from S3 bucket 

In [0]:
Customer= spark.read.option("inferSchema", "true").json("s3://e-commerce-shree/customers_500.json", multiLine=True)
Orders= spark.read.option("inferSchema", "true").json("s3://e-commerce-shree/orders_500.json", multiLine=True)
Payments = spark.read.option("inferSchema", "true").json("s3://e-commerce-shree/payments_500.json", multiLine=True)
Products= spark.read.option("inferSchema", "true").json("s3://e-commerce-shree/products_500.json", multiLine=True)

Customer.show()
Orders.show()
Payments.show()
Products.show()

Customer.write.mode("overwrite").json("/Volumes/workspace/default/shree_project/bronze/customers")
Orders.write.mode("overwrite").json("/Volumes/workspace/default/shree_project/bronze/orders")
Payments.write.mode("overwrite").json("/Volumes/workspace/default/shree_project/bronze/payments")
Products.write.mode("overwrite").json("/Volumes/workspace/default/shree_project/bronze/products")

### Importing basic libraries and transforming the data using pyspark


In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


# df.createOrReplaceTempView("sample")
# spark.sql("select * from sample where order_id=1001").show()
print("====================================================================================================")
Customer.show()

Customer_new=Customer.dropDuplicates(['customer_id']);
Customer_new.show()

c2=Customer_new.withColumn("signup_date",col('signup_date').cast('date'))
c2.printSchema()
cust_final=c2.withColumn("signup_date",date_format(col("signup_date"),"dd-MM-yyyy"))
cust_final.show()


print("====================================================================================================")
Orders.show()
Orders.printSchema()
Orders.select("customer_id").distinct().show()
O1=Orders.filter(col('customer_id')!='null')
ord_final=O1.withColumn("order_date",date_format(col("order_date"),"dd-MM-yyyy"))
ord_final.show()

# # required_cols = ["order_id", "customer_id", "product_id"]

# # o1 = Orders.dropna(subset=required_cols)

# # o1.show()
print("====================================================================================================")

Products.show()
Products.printSchema()
P1=Products.filter(col('product_name')!="null")
P2=P1.withColumn("price", col("price").cast("int"))
prod_final= P2.withColumn("price", round(col("price"), 0))
prod_final.show()

print("====================================================================================================")
Payments.show()
Payments.printSchema()
p1=Payments.withColumn("payment_date",col('payment_date').cast('date'))
p1.printSchema()
p2=p1.withColumn("payment_date",date_format(col("payment_date"),"dd-MM-yyyy"))
p3=p2.withColumn("amount",col('amount').cast('int'))
pay_final=p3.withColumn("amount",round(col('amount'),2))
pay_final.show()

print("====================================================================================================")

### Writing the transformed data to Silver layer(volume) in parquet format


In [0]:
cust_final.write.mode("overwrite") \
  .parquet("/Volumes/workspace/default/shree_project/silver/customers")

ord_final.write.mode("overwrite") \
  .parquet("/Volumes/workspace/default/shree_project/silver/orders")

prod_final.write.mode("overwrite") \
  .parquet("/Volumes/workspace/default/shree_project/silver/products")

pay_final.write.mode("overwrite") \
  .parquet("/Volumes/workspace/default/shree_project/silver/payments")

